# 01. Limpieza y Preparación de Datos (ETL) — E-commerce
Este notebook documenta la auditoría de calidad de datos, estandarización de texto y fechas, ingeniería de variables de negocio y clasificación de ruido operativo para generar el dataset limpio `cleaned_Ecommerce_v2.csv`.

In [ ]:
import pandas as pd
import numpy as np

# Carga del dataset original
df = pd.read_csv('../data/Original_ecommerce.csv', encoding='latin-1')
print(f'Dimensiones iniciales: {df.shape}')

In [ ]:
# Revisión general de los datos
display(df.head(5))
df.info()
display(df.describe())

In [ ]:
# Auditoría de valores nulos
df.isnull().sum()[df.isnull().sum() > 0]

In [ ]:
# Estandarización de cadenas de texto y tratamiento de nulos
text_cols = df.select_dtypes(include=['object', 'string']).columns
df[text_cols] = df[text_cols].apply(lambda x: x.str.strip())

df['Description'] = df['Description'].fillna('UNKNOWN')

# Conversión de fecha transaccional a formato datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

In [ ]:
# Ingeniería de variables de negocio
df['TotalRevenue'] = df['Quantity'] * df['UnitPrice']

df['TransactionType'] = df['InvoiceNo'].astype(str).str.startswith('C').map({
    True: 'Cancellation',
    False: 'Purchase'
})

df['CustomerSegment'] = np.where(df['CustomerID'].notna(), 'Member', 'Guest')

In [ ]:
# Clasificación de ítems: Productos Físicos vs. Ruido de Inventario y Servicios
desc_upper = df['Description'].astype(str).str.upper()
code_upper = df['StockCode'].astype(str).str.upper()

noise_keywords = [
    'DAMAGE', 'DAMAGED', 'LOST', 'CHECK', 'ADJUST',
    'BAD DEBT', 'STOCK', 'AMAZON', 'WRONG', 'SMASHED'
]
service_keywords = [
    'POSTAGE', 'DOTCOM', 'MANUAL', 'CARRIAGE',
    'BANK CHARGES', 'DISCOUNT', 'CRUK', 'PADS TO MATCH'
]
service_codes = ['M', 'POST', 'DOT', 'C2', 'D', 'BANK CHARGES', 'CRUK']

cond_noise = (
    desc_upper.str.contains('|'.join(noise_keywords), regex=True) |
    desc_upper.str.contains(r'\?', regex=True) |
    (df['Quantity'] <= 0)
)
cond_service = (
    desc_upper.str.contains('|'.join(service_keywords), regex=True) |
    code_upper.isin(service_codes)
)

df['ItemCategory'] = np.select(
    [cond_noise, cond_service],
    ['Inventory Adjustment / Noise', 'Service & Shipping'],
    default='Physical Product'
)

print('--- Resumen de Categorización ---')
print(df['ItemCategory'].value_counts())
print(f'Dimensiones finales limpias: {df.shape}')

In [ ]:
# Exportación del dataset limpio listo para Business Intelligence en Excel
df.to_csv('../data/cleaned_Ecommerce_v2.csv', index=False, encoding='utf-8')
print('Archivo cleaned_Ecommerce_v2.csv exportado correctamente.')